# Holter 전체 기록 · 심박 및 HRV 추정 · GPT-6.1-Terra (OpenAI API) 실험

기존의 전압 일반 통계를 빼고 R파와 RR 간격을 이용한 심박수·HRV·감속능 추정치를 생성합니다. `split`이 `development`이면 원본 train 285명과 validation 71명(test 89명과 증강본 제외), `all`이면 test 89명을 포함한 원본 445명을 사용합니다. 공유 가이드의 564명과 다른 코호트입니다.

**연구용 프로토콜:** 정상박동 주석이 없어 RR 필터만으로 진짜 NN을 확인할 수 없으므로, HRV/DC는 임상 검증 전의 추정값입니다. 먼저 로컬 품질 보고서를 확인하세요. 결과는 진료에 사용할 수 없습니다. 계산 방법과 한계는 `Members/sang/docs/holter_llm_feature_method.md`에 있습니다.

이 노트북은 `monorouter_holter_experiment.ipynb`와 같은 방식(특징 계산, 프롬프트, 평가)을 쓰되 모델을 **OpenAI API의 GPT-6.1-Terra**로 호출합니다. 기존 노트북과 결과 폴더는 건드리지 않고 `results/openai_holter/`에 따로 저장합니다.

## 1. 설정

`notebooks/openai_holter.yaml`의 모델·실험·API 설정을 사용합니다. `sample_size: null`은 개발 원본 전체(356명)이며, 숫자를 넣으면 시드 고정 예비 표본입니다. 실제 호출은 `run_live`와 별도로 `clinical.method_validation_confirmed`가 true인 경우에만 가능합니다.

1~3번은 API를 호출하지 않습니다. 파형을 한 번 계산한 뒤 입력과 설정이 같으면 기존 `monorouter_holter` 환자별 캐시를 그대로 재사용하므로, 이미 계산된 개발 환자는 다시 계산하지 않습니다.

`.env` 또는 환경변수에 `OPENAI_API_KEY`를 설정하세요. 키와 `.env` 내용은 출력하지 않습니다. 요청은 `https://api.openai.com/v1`로만 전송합니다. `run_live`를 true로 바꾸면 **Holter 요약 특징이 외부 OpenAI API로 전송**되며 호출 비용이 발생합니다. 환자 ID와 정답은 요청에 포함하지 않습니다.

In [1]:
# ruff: noqa: E402
import hashlib
import json
import math
import os
import re
import urllib.error
import urllib.parse
import urllib.request
from datetime import UTC, datetime
from pathlib import Path
from uuid import uuid4

import numpy as np
import pandas as pd
import yaml
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)


def find_repo():
    for path in (Path.cwd(), *Path.cwd().parents):
        if (path / "Members/sang/model/basic.yaml").is_file():
            return path
    raise RuntimeError("sixletters-poc 저장소 안에서 Jupyter를 실행해 주세요.")


REPO = find_repo()
import sys

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from Members.sang.experiments.monorouter_holter.clinical_features import (
    prepare_development,
    reference_quality_report,
    select_development,
    validate_llm_features,
)
from Members.sang.model.config import load_basic_config, resolve_data_path


def read_env(path):
    """Read the simple KEY=value format used by this notebook's .env.example."""
    values = {}
    if not path.is_file():
        return values
    for line_no, line in enumerate(path.read_text(encoding="utf-8").splitlines(), 1):
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        if "=" not in line:
            raise ValueError(f".env {line_no}행: KEY=value 형식이 아닙니다.")
        key, value = line.split("=", 1)
        key, value = key.strip(), value.strip()
        if not key.replace("_", "").isalnum():
            raise ValueError(f".env {line_no}행: 환경변수 이름이 잘못됐습니다.")
        if len(value) >= 2 and value[0] == value[-1] and value[0] in (chr(34), chr(39)):
            value = value[1:-1]
        values[key] = value
    return values


file_env = read_env(REPO / "Members/sang/notebooks/.env")


def setting(name, default=""):
    return os.environ.get(name) or file_env.get(name) or default


API_KEY = setting("OPENAI_API_KEY")
BASE_URL = "https://api.openai.com/v1"
parsed = urllib.parse.urlsplit(BASE_URL)
if (
    parsed.scheme != "https"
    or parsed.hostname != "api.openai.com"
    or parsed.username
    or parsed.password
    or parsed.port not in (None, 443)
    or parsed.path != "/v1"
    or parsed.query
    or parsed.fragment
):
    raise ValueError("BASE_URL은 https://api.openai.com/v1 이어야 합니다.")
print("프로젝트:", REPO)
print("API 키:", "설정됨" if API_KEY else "미설정")

프로젝트: /home/sang/sixletters-poc
API 키: 설정됨


In [2]:
CONFIG_PATH = REPO / "Members/sang/notebooks/openai_holter.yaml"
config = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
MODELS = config["models"]
experiment, api_config, clinical = config["experiment"], config["api"], config["clinical"]
if (
    not isinstance(MODELS, list)
    or not MODELS
    or any(not isinstance(x, str) or not x.strip() or x != x.strip() for x in MODELS)
    or len(set(MODELS)) != len(MODELS)
):
    raise ValueError("중복이나 공백이 없는 모델 ID 목록이 필요합니다.")
SPLIT = experiment["split"]
SAMPLE_SIZE, SAMPLE_SEED = experiment["sample_size"], experiment["sample_seed"]
THRESHOLD, RUN_LIVE = experiment["threshold"], experiment["run_live"]
METHOD_VALIDATED = clinical["method_validation_confirmed"]
SETTINGS = {
    k: clinical[k]
    for k in (
        "chunk_seconds",
        "context_seconds",
        "min_rr_ms",
        "max_rr_ms",
        "relative_limit",
        "sdann_min_coverage",
        "hourly_min_coverage",
    )
}
MAX_COMPLETION_TOKENS = api_config["max_completion_tokens"]
REASONING_EFFORT, TIMEOUT = api_config["reasoning_effort"], api_config["timeout_seconds"]
MAX_ATTEMPTS = api_config.get(
    "max_attempts", 1
)  # 응답이 null이거나 형식이 틀리면 같은 요청을 이 횟수까지 시도합니다.
if type(MAX_ATTEMPTS) is not int or not 1 <= MAX_ATTEMPTS <= 5:
    raise ValueError("max_attempts는 1~5의 정수여야 합니다.")
if SPLIT not in {"development", "all"}:
    raise ValueError("split은 development 또는 all이어야 합니다.")
if SAMPLE_SIZE is not None and (type(SAMPLE_SIZE) is not int or SAMPLE_SIZE < 1):
    raise ValueError("sample_size를 확인하세요.")
if type(SAMPLE_SEED) is not int or type(RUN_LIVE) is not bool or type(METHOD_VALIDATED) is not bool:
    raise ValueError("실험 설정을 확인하세요.")
if type(THRESHOLD) not in (int, float) or not math.isfinite(THRESHOLD) or not 0 <= THRESHOLD <= 1:
    raise ValueError("threshold를 확인하세요.")
if (
    type(MAX_COMPLETION_TOKENS) is not int
    or MAX_COMPLETION_TOKENS < 1
    or REASONING_EFFORT not in {None, "low", "medium", "high"}
    or type(TIMEOUT) not in (int, float)
    or not math.isfinite(TIMEOUT)
    or TIMEOUT <= 0
):
    raise ValueError("API 설정을 확인하세요.")
if (
    type(SETTINGS["chunk_seconds"]) is not int
    or SETTINGS["chunk_seconds"] < 30
    or SETTINGS["chunk_seconds"] % 30
    or type(SETTINGS["context_seconds"]) is not int
    or not 0 <= SETTINGS["context_seconds"] <= 30
):
    raise ValueError("chunk/context 설정을 확인하세요.")
if (
    any(
        type(SETTINGS[k]) not in (int, float) or not math.isfinite(SETTINGS[k])
        for k in (
            "min_rr_ms",
            "max_rr_ms",
            "relative_limit",
            "sdann_min_coverage",
            "hourly_min_coverage",
        )
    )
    or not 0 < SETTINGS["min_rr_ms"] < SETTINGS["max_rr_ms"]
    or not 0 < SETTINGS["relative_limit"] < 1
    or not all(0 < SETTINGS[k] <= 1 for k in ("sdann_min_coverage", "hourly_min_coverage"))
):
    raise ValueError("RR 또는 품질 설정을 확인하세요.")
REFERENCE_PATH = REPO / "Members/sang/datas/Holter/physionet_reference/rhythm_hrv_dev.csv"
print("모델:", MODELS, "| 표본:", SAMPLE_SIZE if SAMPLE_SIZE is not None else "개발 전체")
print("호출 설정:", RUN_LIVE, "| 지표 검증 확인:", METHOD_VALIDATED)

모델: ['gpt-6.1-terra'] | 표본: 개발 전체
호출 설정: True | 지표 검증 확인: True


In [ ]:
# 전체(train+validation+test) 실행용 선택기. 기본 선택기는 test를 막아 두므로 split이 "all"일 때만 대체합니다.
import csv

import Members.sang.experiments.monorouter_holter.clinical_features as holter_cf

REFERENCE_CSVS = [
    REPO / "Members/sang/datas/Holter/physionet_reference/rhythm_hrv_dev.csv",
    REPO / "Members/sang/datas/Holter/physionet_reference/rhythm_hrv_test.csv",
]


def select_all_original(split_path, wave_dir, reference_path, sample_size, seed):
    """원본 train·validation·test 전체를 고릅니다. 증강 복사본(-1, -2)은 제외합니다."""
    if sample_size is not None:
        raise ValueError("all 분할에서는 sample_size를 null로 두세요.")
    reference = {}
    for path in REFERENCE_CSVS:
        with path.open(encoding="utf-8-sig", newline="") as stream:
            for row in csv.DictReader(stream):
                pid = row["patient_id"].strip()
                if pid in reference:
                    raise ValueError("리듬 참고 정보에 중복 ID가 있습니다.")
                reference[pid] = row
    records = []
    for pid, group, label in holter_cf.read_patient_split(Path(split_path)):
        if group not in {"train", "validation", "test"} or re.search(r"-\d+$", pid):
            continue
        wave_path = Path(wave_dir) / f"{pid}.npy"
        if not wave_path.is_file():
            raise FileNotFoundError("선택한 원본 파형이 없습니다. 데이터 경로를 확인하세요.")
        if pid not in reference:
            raise ValueError("선택한 기록의 리듬 참고 정보가 없습니다.")
        records.append(
            {
                "patient_id": pid,
                "split": group,
                "label": int(label),
                "wave_path": wave_path,
                "reference": reference[pid],
            }
        )
    if len(records) != 445 or len({r["patient_id"] for r in records}) != 445:
        raise ValueError(f"전체 원본 환자는 445명이어야 합니다: {len(records)}명")
    return records


if SPLIT == "all":
    holter_cf.select_development = select_all_original
    select_development = select_all_original
    print("전체 모드: 원본 445명(train·validation·test)을 대상으로 합니다.")

## 2. 개발 데이터 선택

기존 Holter 분할에서 원본 train·validation만 선택하고, 증강 파일과 test를 제외합니다. PhysioNet의 리듬 코드는 HRV 적용 여부를 정할 때만 사용합니다. 이 단계는 파형 전체나 API를 읽지 않습니다.

In [3]:
basic_path = REPO / "Members/sang/model/basic.yaml"
basic = load_basic_config(basic_path)
raw = basic["data"]["raw_ecg"]
selected = select_development(
    resolve_data_path(raw["split_path"], basic_path),
    resolve_data_path(raw["wave_directory"], basic_path),
    REFERENCE_PATH,
    SAMPLE_SIZE,
    SAMPLE_SEED,
)
print(
    "선택한 개발 원본:",
    len(selected),
    "명 | 증강본 제외" + (" (test 포함)" if SPLIT == "all" else " | test 제외"),
)

선택한 개발 원본: 356 명 | test 및 증강본 제외


## 3. 전체 기록 특징 계산과 로컬 품질 확인

30초 품질 제외 구간을 적용하고 유도별 R파를 검출합니다. 선택된 유도의 RR 간격에서 심박수와 HRV/DC **추정치**를 구합니다. 심방세동·그 외 동율동 미확인 기록은 HRV/DC를 비워 둡니다. 비정상박동 주석이 없으므로 추정치가 검증된 NN 지표라는 뜻은 아닙니다.

아래 참고값 비교는 계산 검토용이며 정답 라벨을 사용하지 않습니다. 소수 환자의 차이만으로 검증 완료라고 판단할 수 없습니다. 처음에는 시간이 오래 걸릴 수 있고, 재실행 시 환자별 캐시를 사용합니다.

In [4]:
selected, prepared = prepare_development(
    REPO, basic, basic_path, REFERENCE_PATH, SAMPLE_SIZE, SAMPLE_SEED, SETTINGS
)
valid = [(pid, label, f) for pid, label, f, error in prepared if f is not None]
print("특징 계산 성공:", len(valid), "/", len(prepared))
if not valid:
    raise RuntimeError("유효한 특징이 없습니다. 입력과 오류를 확인하세요.")
print("참고 HRV와 절대 차이 (동일 지표 계산법이라는 보장은 없음):")
display(pd.DataFrame(reference_quality_report(selected, prepared)).T)
print("리듬 및 분석 시간 요약:")
display(
    pd.DataFrame(
        [
            {
                "rhythm": f["recording"]["holter_rhythm"],
                "hours": f["recording"]["analyzed_hours"],
                "hrv_status": f["quality"]["hrv_status"],
            }
            for _, _, f in valid
        ]
    )
    .groupby(["rhythm", "hrv_status"], dropna=False)
    .agg(n=("hours", "size"), median_hours=("hours", "median"))
)
# 이 파일은 로컬 개인별 결과이며 API 입력에는 환자 ID와 정답을 포함하지 않습니다.
feature_dir = (
    REPO
    / "Members/sang/results/openai_holter"
    / ("features_" + datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ_") + uuid4().hex[:8])
)
feature_dir.mkdir(mode=0o700, parents=True, exist_ok=False)
feature_file = feature_dir / "features_by_patient.json"
feature_file.write_text(
    json.dumps({pid: f for pid, _, f in valid}, ensure_ascii=False, indent=2, allow_nan=False),
    encoding="utf-8",
)
os.chmod(feature_file, 0o600)
feature_csv = feature_dir / "features_by_patient.csv"
pd.json_normalize([{"patient_id": pid, **f} for pid, _, f in valid], sep=".").to_csv(
    feature_csv, index=False
)
os.chmod(feature_csv, 0o600)
print("로컬 특징 파일:", feature_dir)

특징 준비 1/356: 캐시
특징 준비 2/356: 캐시
특징 준비 3/356: 캐시
특징 준비 4/356: 캐시
특징 준비 5/356: 캐시
특징 준비 6/356: 캐시
특징 준비 7/356: 캐시
특징 준비 8/356: 캐시
특징 준비 9/356: 캐시
특징 준비 10/356: 캐시
특징 준비 11/356: 캐시
특징 준비 12/356: 캐시
특징 준비 13/356: 캐시
특징 준비 14/356: 캐시
특징 준비 15/356: 캐시
특징 준비 16/356: 캐시
특징 준비 17/356: 캐시
특징 준비 18/356: 캐시
특징 준비 19/356: 캐시
특징 준비 20/356: 캐시
특징 준비 21/356: 캐시
특징 준비 22/356: 캐시
특징 준비 23/356: 캐시
특징 준비 24/356: 캐시
특징 준비 25/356: 캐시
특징 준비 26/356: 캐시
특징 준비 27/356: 캐시
특징 준비 28/356: 캐시
특징 준비 29/356: 캐시
특징 준비 30/356: 캐시
특징 준비 31/356: 캐시
특징 준비 32/356: 캐시
특징 준비 33/356: 캐시
특징 준비 34/356: 캐시
특징 준비 35/356: 캐시
특징 준비 36/356: 캐시
특징 준비 37/356: 캐시
특징 준비 38/356: 캐시
특징 준비 39/356: 캐시
특징 준비 40/356: 캐시
특징 준비 41/356: 캐시
특징 준비 42/356: 캐시
특징 준비 43/356: 캐시
특징 준비 44/356: 캐시
특징 준비 45/356: 캐시
특징 준비 46/356: 캐시
특징 준비 47/356: 캐시
특징 준비 48/356: 캐시
특징 준비 49/356: 캐시
특징 준비 50/356: 캐시
특징 준비 51/356: 캐시
특징 준비 52/356: 캐시
특징 준비 53/356: 캐시
특징 준비 54/356: 캐시
특징 준비 55/356: 캐시
특징 준비 56/356: 캐시
특징 준비 57/356: 캐시
특징 준비 58/356: 캐시
특징 준비 59/356: 캐시
특징 준비 

,matched,median_absolute_difference,mean_absolute_difference
SDNN_ms,263.0,1.556656,4.920669
SDANN_ms,263.0,1.097524,4.260915
RMSSD_ms,263.0,3.355023,7.583157
pNN50_percent,263.0,0.668504,2.451362


리듬 및 분석 시간 요약:


,,n,median_hours
rhythm,hrv_status,,
permanent_atrial_fibrillation,not_available_non_sinus_or_unknown,63,22.8500
sinus,experimental_rr_surrogate,290,23.6583
unknown,not_available_non_sinus_or_unknown,3,23.9833


로컬 특징 파일: /home/sang/sixletters-poc/Members/sang/results/openai_holter/features_20261104T030151Z_3966a193


## 4. 모델 요청 및 응답 검증

모델에는 검증된 필드만 JSON으로 전달합니다. 환자 ID, SCD 정답, 참고 HRV 값은 전송하지 않습니다. 응답은 0~100%의 단일 숫자로 검증합니다. 임상 정보 27개 실험은 해당 데이터와 같은 환자 매칭을 확인한 후 별도로 구성해야 합니다.

In [5]:
SYSTEM_PROMPT = (
    "This is retrospective research, not clinical care. Given experimental Holter-derived RR and HRV estimates, "
    "estimate the percentage probability of sudden cardiac death within four years. "
    "Normal beat classification and these features are not validated; null means unavailable. "
    "Do not infer identities or invent missing measurements. "
    "Return only a JSON object with the single key risk_percent and a finite number from 0 to 100."
)


class ApiError(RuntimeError):
    def __init__(self, status):
        self.status = status
        super().__init__(f"OpenAI HTTP {status}")


class NoRedirect(urllib.request.HTTPRedirectHandler):
    def redirect_request(self, req, fp, code, msg, headers, newurl):
        return None


def make_request_body(features, model):
    validate_llm_features(features)
    user_message = json.dumps(
        {"ecg_summary": features}, ensure_ascii=True, allow_nan=False, separators=(",", ":")
    )
    body = {
        "model": model,
        "max_completion_tokens": MAX_COMPLETION_TOKENS,
        "stream": False,
        "response_format": {"type": "json_object"},
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_message},
        ],
    }
    if REASONING_EFFORT is not None:
        body["reasoning_effort"] = REASONING_EFFORT
    return body


def parse_response(data):
    if not isinstance(data, dict):
        raise ValueError("API 응답 형식이 잘못됐습니다.")
    choices = data.get("choices")
    if not isinstance(choices, list) or len(choices) != 1 or not isinstance(choices[0], dict):
        raise ValueError("단일 모델 응답이 아닙니다.")
    choice = choices[0]
    if choice.get("finish_reason") != "stop":
        raise ValueError("모델 응답이 정상 종료되지 않았습니다. 출력 토큰 한도를 확인하세요.")
    message = choice.get("message")
    if (
        not isinstance(message, dict)
        or message.get("refusal")
        or not isinstance(message.get("content"), str)
    ):
        raise ValueError("모델이 요청을 거절했거나 텍스트 응답이 없습니다.")
    result = json.loads(message["content"])
    if not isinstance(result, dict) or set(result) != {"risk_percent"}:
        raise ValueError("출력 JSON 스키마가 다릅니다.")
    probability = result["risk_percent"]
    if (
        type(probability) not in (int, float)
        or not math.isfinite(probability)
        or not 0 <= probability <= 100
    ):
        raise ValueError("risk_percent는 0~100의 유한 숫자여야 합니다.")
    usage = data.get("usage") or {}
    if not isinstance(usage, dict):
        raise ValueError("사용량 응답 형식이 잘못됐습니다.")
    model_returned = data.get("model")
    if model_returned is not None and not isinstance(model_returned, str):
        raise ValueError("반환된 모델 ID 형식이 잘못됐습니다.")
    return float(probability) / 100, {
        "input_tokens": usage.get("prompt_tokens"),
        "output_tokens": usage.get("completion_tokens"),
        "model_returned": model_returned,
    }


def predict_one(features, model):
    if not API_KEY:
        raise RuntimeError("OPENAI_API_KEY를 .env 또는 환경변수에 설정하세요.")
    payload = json.dumps(make_request_body(features, model), allow_nan=False).encode("utf-8")
    opener = urllib.request.build_opener(NoRedirect())

    req = urllib.request.Request(
        BASE_URL + "/chat/completions",
        data=payload,
        method="POST",
        headers={
            "Content-Type": "application/json",
            "Authorization": "Bearer " + API_KEY,
        },
    )
    try:
        with opener.open(req, timeout=TIMEOUT) as response:
            data = json.load(response)
    except urllib.error.HTTPError as exc:
        raise ApiError(exc.code) from None
    except urllib.error.URLError as exc:
        raise RuntimeError(f"OpenAI 연결 실패: {type(exc.reason).__name__}") from None
    return parse_response(data)


# 네트워크 없이 GPT 응답 검증 경로를 점검합니다.
example = {"choices": [{"finish_reason": "stop", "message": {"content": '{"risk_percent":25}'}}]}
assert parse_response(example)[0] == 0.25
try:
    parse_response(
        {"choices": [{"finish_reason": "stop", "message": {"content": '{"risk_percent":150}'}}]}
    )
except ValueError:
    pass
else:
    raise AssertionError("범위 밖 확률이 거부되지 않았습니다.")
print("GPT 요청 구성과 응답 검증 준비 완료")

GPT 요청 구성과 응답 검증 준비 완료


## 5. 모델별 예측

로컬 지표 검증 후 YAML의 `clinical.method_validation_confirmed`와 `experiment.run_live`가 모두 true일 때만 API를 호출합니다. 이전 실험의 `run_live` 설정은 유지했지만 새 지표 검증 전에는 호출되지 않습니다. 인증 오류·요청 제한 시 나머지 호출을 중단합니다.

기존 실험 경로가 YAML에 지정되면 모델·계산 설정·환자별 특징과 라벨을 대조한 후 일치하는 성공 응답만 재사용합니다.

In [6]:
results = {}
result_dir = None
if not RUN_LIVE or not METHOD_VALIDATED:
    body = make_request_body(valid[0][2], MODELS[0])
    if RUN_LIVE and not METHOD_VALIDATED:
        print("새 지표 검증 전이므로 호출을 보류합니다.")
    print("DRY RUN: 네트워크 요청 없음 | 대상:", BASE_URL)
    print(
        "모델:",
        len(MODELS),
        "| 모델당 표본:",
        len(prepared),
        "| 예정 요청:",
        len(MODELS) * len(prepared),
    )
    print("예시 요청 바이트:", len(json.dumps(body).encode("utf-8")))
else:
    if not API_KEY:
        raise RuntimeError("실제 호출 전 OPENAI_API_KEY를 설정하세요.")
    result_dir = (
        REPO
        / "Members/sang/results/openai_holter"
        / (datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ_") + uuid4().hex[:8])
    )
    result_dir.mkdir(mode=0o700, parents=True, exist_ok=False)
    metadata = {
        "split": SPLIT,
        "sample_size": len(prepared),
        "sample_seed": SAMPLE_SEED,
        "clinical_settings": SETTINGS,
        "threshold": THRESHOLD,
        "models": MODELS,
        "max_completion_tokens": MAX_COMPLETION_TOKENS,
        "reasoning_effort": REASONING_EFFORT,
        "max_attempts": MAX_ATTEMPTS,
        "api_protocol": "openai_chat_completions",
        "provider": "api.openai.com",
        "feature_version": "experimental_rr_v1",
        "status": "RUNNING",
    }
    # 모델 ID가 실제로 존재하는지 목록 조회로 먼저 확인합니다(추론·데이터 전송 없음).
    list_request = urllib.request.Request(
        BASE_URL + "/models", headers={"Authorization": "Bearer " + API_KEY}
    )
    try:
        with urllib.request.build_opener(NoRedirect()).open(list_request, timeout=TIMEOUT) as r:
            available = {m["id"] for m in json.load(r)["data"]}
    except urllib.error.HTTPError as exc:
        raise RuntimeError(f"모델 목록 조회 실패: HTTP {exc.code}") from None
    missing = [m for m in MODELS if m not in available]
    if missing:
        raise ValueError(f"이 API 키로 사용할 수 없는 모델 ID입니다: {missing}")
    print("모델 ID 확인 완료:", MODELS)
    metadata_path = result_dir / "metadata.json"
    metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    # 기존 예측은 동일 모델·설정·라벨·특징에 한해서만 재사용합니다.
    reusable = {}
    prior_dir, prior_features_path = (
        experiment.get("reuse_predictions_dir"),
        experiment.get("reuse_features_file"),
    )
    if prior_dir and prior_features_path:
        old_dir, old_features = REPO / prior_dir, REPO / prior_features_path
        old_metadata = json.loads((old_dir / "metadata.json").read_text(encoding="utf-8"))
        comparable = (
            old_metadata.get("status") == "COMPLETE"
            and old_metadata.get("split") == SPLIT
            and old_metadata.get("feature_version") == "experimental_rr_v1"
            and old_metadata.get("clinical_settings") == SETTINGS
            and old_metadata.get("models") == MODELS
            and old_metadata.get("max_completion_tokens") == MAX_COMPLETION_TOKENS
            and old_metadata.get("reasoning_effort") == REASONING_EFFORT
        )
        if not comparable:
            raise ValueError("기존 실험 설정이 다르므로 재사용할 수 없습니다.")
        feature_map = json.loads(old_features.read_text(encoding="utf-8"))
        old_files = list(old_dir.glob("*/predictions.csv"))
        if len(old_files) != 1:
            raise ValueError("재사용 예측 파일이 하나여야 합니다.")
        previous = pd.read_csv(old_files[0])
        if previous.patient_id.duplicated().any():
            raise ValueError("기존 예측에 중복 환자가 있습니다.")
        previous = {
            r["patient_id"]: r for r in previous.to_dict("records") if r.get("status") == "ok"
        }
        for pid, target, features, _ in prepared:
            if pid in previous:
                if (
                    features is None
                    or feature_map.get(pid) != features
                    or int(previous[pid]["y_true"]) != target
                ):
                    raise ValueError(
                        "기존 예측과 현재 환자의 특징 또는 정답이 달라 재사용을 중단합니다."
                    )
                reusable[pid] = previous[pid]
        metadata["reused_predictions"] = len(reusable)
        metadata["reused_from"] = str(prior_dir)
        metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
        print("동일 환자·특징의 기존 성공 예측 재사용:", len(reusable), "건")
    stop_all = False
    for model_index, model in enumerate(MODELS, 1):
        if stop_all:
            break
        records = []
        digest = hashlib.sha256(model.encode("utf-8")).hexdigest()[:8]
        slug = re.sub(r"[^a-zA-Z0-9_.-]+", "_", model)[:80]
        model_dir = result_dir / f"{slug}_{digest}"
        model_dir.mkdir(mode=0o700)
        for index, (pid, target, features, _feature_error) in enumerate(prepared):
            row = {
                "patient_id": pid,
                "y_true": target,
                "p_positive": None,
                "input_tokens": None,
                "output_tokens": None,
                "model_returned": None,
                "status": "error",
                "error_type": None,
                "http_status": None,
            }
            try:
                if pid in reusable:
                    row.update(reusable[pid])
                    row["source"] = "reused"
                else:
                    if features is None:
                        raise ValueError("특징 계산 실패")
                    for attempt in range(MAX_ATTEMPTS):
                        try:
                            probability, usage = predict_one(features, model)
                            break
                        except ValueError:
                            if attempt + 1 == MAX_ATTEMPTS:
                                raise
                    row.update(p_positive=probability, **usage, status="ok", source="new")
            except (RuntimeError, ValueError, KeyError, TypeError, TimeoutError) as exc:
                row["error_type"] = type(exc).__name__
                if isinstance(exc, ApiError):
                    row["http_status"] = exc.status
                    if exc.status in (400, 401, 402, 403, 404, 422, 429):
                        stop_all = True
                print(
                    f"모델 {model_index}/{len(MODELS)}, 표본 {index + 1}/{len(prepared)}: {type(exc).__name__}"
                    + (f" HTTP {exc.status}" if isinstance(exc, ApiError) else "")
                )
            records.append(row)
            temp = model_dir / "predictions.csv.tmp"
            pd.DataFrame(records).to_csv(temp, index=False)
            os.chmod(temp, 0o600)
            temp.replace(model_dir / "predictions.csv")
            if stop_all:
                print(
                    "요청 형식·인증·크레딧·접근 또는 요청 제한 오류로 나머지 요청을 중단했습니다."
                )
                break
            print(f"모델 {model_index}/{len(MODELS)}, 표본 {index + 1}/{len(prepared)} 처리 완료")
        results[model] = {"records": records, "directory": model_dir}
    metadata["status"] = (
        "COMPLETE"
        if len(results) == len(MODELS)
        and all(
            len(v["records"]) == len(prepared) and all(r["status"] == "ok" for r in v["records"])
            for v in results.values()
        )
        else "PARTIAL"
    )
    metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    print("로컬 결과:", result_dir, "| 상태:", metadata["status"])

모델 ID 확인 완료: ['gpt-6.1-terra']
모델 1/1, 표본 1/356 처리 완료
모델 1/1, 표본 2/356 처리 완료
모델 1/1, 표본 3/356 처리 완료
모델 1/1, 표본 4/356 처리 완료
모델 1/1, 표본 5/356 처리 완료
모델 1/1, 표본 6/356 처리 완료
모델 1/1, 표본 7/356 처리 완료
모델 1/1, 표본 8/356 처리 완료
모델 1/1, 표본 9/356: ValueError
모델 1/1, 표본 9/356 처리 완료
모델 1/1, 표본 10/356 처리 완료
모델 1/1, 표본 11/356 처리 완료
모델 1/1, 표본 12/356 처리 완료
모델 1/1, 표본 13/356 처리 완료
모델 1/1, 표본 14/356 처리 완료
모델 1/1, 표본 15/356 처리 완료
모델 1/1, 표본 16/356 처리 완료
모델 1/1, 표본 17/356 처리 완료
모델 1/1, 표본 18/356 처리 완료
모델 1/1, 표본 19/356 처리 완료
모델 1/1, 표본 20/356 처리 완료
모델 1/1, 표본 21/356 처리 완료
모델 1/1, 표본 22/356 처리 완료
모델 1/1, 표본 23/356: ValueError
모델 1/1, 표본 23/356 처리 완료
모델 1/1, 표본 24/356 처리 완료
모델 1/1, 표본 25/356 처리 완료
모델 1/1, 표본 26/356 처리 완료
모델 1/1, 표본 27/356 처리 완료
모델 1/1, 표본 28/356: ValueError
모델 1/1, 표본 28/356 처리 완료
모델 1/1, 표본 29/356 처리 완료
모델 1/1, 표본 30/356 처리 완료
모델 1/1, 표본 31/356 처리 완료
모델 1/1, 표본 32/356 처리 완료
모델 1/1, 표본 33/356 처리 완료
모델 1/1, 표본 34/356: ValueError
모델 1/1, 표본 34/356 처리 완료
모델 1/1, 표본 35/356: ValueError
모델 1/1, 표본 35

## 6. 로컬 평가

AUROC·AP와 고정 임계값의 혼동행렬을 표시합니다. 20명 예비 표본의 수치는 결론이나 기존 564명 결과와 비교하는 용도가 아닙니다. 전체 개발자 중 동일 환자만 비교해야 하며, DL과 동일한 5개 fold 파일이 이 저장소에 확인되지 않아 로지스틱 OOF 비교는 아직 제공하지 않습니다. test는 접근하지 않습니다.

In [7]:
if not results:
    print("평가할 API 응답이 없습니다. DRY RUN에서는 요청하지 않습니다.")
else:
    comparison = []
    for model, output in results.items():
        predictions = pd.DataFrame(output["records"])
        successful = predictions.loc[predictions.status == "ok"].copy()
        metrics = {
            "model": model,
            "split": SPLIT,
            "expected": len(prepared),
            "attempted": len(predictions),
            "succeeded": len(successful),
            "failed": len(predictions) - len(successful),
            "threshold": THRESHOLD,
            "complete": len(successful) == len(prepared),
            "note": "sample-only; may be biased by sampling or failed requests",
        }
        if not successful.empty:
            y = successful.y_true.astype(int).to_numpy()
            probability = successful.p_positive.astype(float).to_numpy()
            pred = (probability >= THRESHOLD).astype(int)
            tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel().tolist()
            metrics.update(
                positive=int(y.sum()),
                negative=int((1 - y).sum()),
                accuracy=float(accuracy_score(y, pred)),
                recall=float(tp / (tp + fn)) if tp + fn else None,
                specificity=float(tn / (tn + fp)) if tn + fp else None,
                confusion_matrix=[[tn, fp], [fn, tp]],
                average_precision=float(average_precision_score(y, probability))
                if len(np.unique(y)) == 2
                else None,
                auroc=float(roc_auc_score(y, probability)) if len(np.unique(y)) == 2 else None,
            )
        if not successful.empty and len(np.unique(y)) == 2:
            fpr, tpr, thresholds = roc_curve(y, probability, drop_intermediate=False)
            allowed = np.flatnonzero(tpr >= 0.8)
            if len(allowed):
                best = allowed[np.argmin(fpr[allowed])]
                metrics["specificity_at_sensitivity_ge_0_8"] = float(1 - fpr[best])
                metrics["sensitivity_at_selected_threshold"] = float(tpr[best])
                metrics["selected_threshold_for_sensitivity"] = float(thresholds[best])
                metrics["threshold_note"] = (
                    "descriptive threshold chosen on same data; not an unbiased deployment estimate"
                )
        (output["directory"] / "metrics.json").write_text(
            json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8"
        )
        comparison.append(metrics)
    comparison_frame = pd.DataFrame(comparison)
    comparison_frame.to_csv(result_dir / "model_comparison.csv", index=False)
    display(comparison_frame)
    if any(not row["complete"] for row in comparison) or len(results) != len(MODELS):
        print("일부 모델 또는 표본이 누락되어 결과를 동등하게 비교할 수 없습니다.")

,model,split,expected,attempted,succeeded,failed,threshold,complete,note,positive,...,accuracy,recall,specificity,confusion_matrix,average_precision,auroc,specificity_at_sensitivity_ge_0_8,sensitivity_at_selected_threshold,selected_threshold_for_sensitivity,threshold_note
0,gpt-6.1-terra,development,356,356,278,78,0.5,False,sample-only; may be biased by sampling or fail...,28,...,0.899281,0.0,1.0,"[[250, 0], [28, 0]]",0.098724,0.4575,0.032,0.964286,0.004,descriptive threshold chosen on same data; not...


일부 모델 또는 표본이 누락되어 결과를 동등하게 비교할 수 없습니다.
